In [ ]:
!apt-get install poppler-utils
!pip install pdf2image requests tqdm

In [ ]:
import os
import base64
import requests
from pdf2image import convert_from_path
from tqdm import tqdm
from google.colab import userdata

# ==============================
# CONFIGURATION - Update these variables as needed
# ==============================

PDF_PATH = "/content/academic calendar year 2025-2026.pdf"
DOC_ID = "03"
FILE_NAME = "Academic Calendar for Undergraduate Studies Session 2025-2026 (Updated)"
SOURCE_LINK = "https://www.bpps.unimas.my/media/attachments/2025/12/31/kalendar-akademik-pengajian-ijazah-sarjana-muda-sesi-2025_pindaan-171202025.pdf"
OUTPUT_MD = "/content/academic_calendar_ug_content2.md"
TEMP_IMAGE_DIR = "/content/pdf_pages"

MODEL_NAME = "qwen/qwen3.5-397b-a17b"
INVOKE_URL = "https://integrate.api.nvidia.com/v1/chat/completions"
stream = False

NVIDIA_API_KEY = userdata.get("NVIDIA_NIM_API")

HEADERS = {
    "Authorization": f"Bearer {NVIDIA_API_KEY}",
    "Accept": "application/json"
}

QUERY_PROMPT = """
The provided image is a page from a university academic calendar document.

Your task:
- Read the FULL page carefully.
- Extract the COMPLETE textual representation.
- Preserve headings and hierarchy.
- Format output in clean Markdown.
- Preserve tables using proper Markdown table syntax.
- Do NOT summarize.
- Do NOT infer missing information.
- Do NOT add content not visible in the image.
- Maintain structural accuracy.
- Ensure the output is comprehensive and faithful to the original page.

Return only the Markdown representation.
"""

In [ ]:
# ==============================
# STEP 1 – Convert PDF to Images
# ==============================

if not os.path.exists(TEMP_IMAGE_DIR):
    os.makedirs(TEMP_IMAGE_DIR)

print("Converting PDF to images...")

pages = convert_from_path(PDF_PATH, dpi=300)

image_paths = []
for i, page in enumerate(pages):
    image_path = os.path.join(TEMP_IMAGE_DIR, f"page_{i+1}.png")
    page.save(image_path, "PNG")
    image_paths.append(image_path)

print(f"Converted {len(image_paths)} pages successfully.")

In [ ]:
# ==============================
# Helper Functions
# ==============================

def encode_image_to_base64(path):
    with open(path, "rb") as f:
        return base64.b64encode(f.read()).decode()

def send_to_vlm(image_path):
    b64_image = encode_image_to_base64(image_path)

    content_payload = [
        {
            "type": "image_url",
            "image_url": {
                "url": f"data:image/png;base64,{b64_image}"
            }
        },
        {
            "type": "text",
            "text": QUERY_PROMPT
        }
    ]

    payload = {
        "model": MODEL_NAME,
        "messages": [{"role": "user", "content": content_payload}],
        "max_tokens": 16384,
        "temperature": 0.2,
        "top_p": 0.80,
        "top_k": 20,
        "presence_penalty": 0,
        "repetition_penalty": 1,
        "stream": stream,
        "chat_template_kwargs": {"enable_thinking": False}
    }

    response = requests.post(INVOKE_URL, headers=HEADERS, json=payload, stream=stream)

    return response

In [ ]:
import time

# ==============================
# STEP 2 – Loop Through Pages
# ==============================

print("\nStarting VLM extraction...\n")

for idx, image_path in enumerate(tqdm(image_paths)):

    page_number = idx + 1
    print(f"\nProcessing Page {page_number}...")

    # Add a delay before processing each new page
    time.sleep(15)

    retry_attempts = 0
    MAX_RETRIES = 5

    while True:
        response = send_to_vlm(image_path)

        if response.status_code == 200:
            content = response.json()["choices"][0]["message"]["content"]

            # Add page separator
            formatted_page = (
                f"\n\n===\n\nPhysical Page {page_number}\nDoc ID: {DOC_ID}\n"
                f"Source: {FILE_NAME}\nLink: {SOURCE_LINK} \n\n{content}\n"
            )

            with open(OUTPUT_MD, "a", encoding="utf-8") as f:
                f.write(formatted_page)

            print(f"Page {page_number} extracted successfully.")
            break

        else:
            print(f"\nError on Page {page_number}")
            print(f"Status Code: {response.status_code}")
            print(response.text)

            retry_attempts += 1
            if retry_attempts <= MAX_RETRIES:
                print(f"Retrying (Attempt {retry_attempts}/{MAX_RETRIES})... Waiting 30 seconds before retry.")
                time.sleep(30) # Delay before retrying
                continue
            else:
                user_input = input("Max retries reached. Retry this page again? (y/n): ").strip().lower()

                if user_input == "y":
                    print("Retrying... Waiting 30 seconds before retry.")
                    time.sleep(30) # Delay before retrying again after user input
                    retry_attempts = 0 # Reset retry count if user chooses to retry
                    continue
                else:
                    print("Extraction terminated by user.")
                    exit()

# ==============================
# STEP 3 – Write Final Markdown
# ==============================

print("\nAll pages extracted successfully.")
print(f"Markdown saved to: {OUTPUT_MD}")